# DSA 210 Final Report
# Predicting Agricultural Crop Yield Across Turkish Provinces Using Weather and Environmental Data

**Course:** DSA 210 — Introduction to Data Science, Spring 2026  
**Date:** May 2026

---

## Table of Contents
1. [Motivation](#1-motivation)
2. [Data Source](#2-data-source)
3. [Data Analysis](#3-data-analysis)
4. [Findings](#4-findings)
5. [Limitations and Future Work](#5-limitations-and-future-work)

## 1. Motivation

Agriculture is one of the most climate-sensitive sectors of the economy. Turkey, as one of the world's top ten agricultural producers in wheat, barley, cotton, and various fruits, faces a unique challenge: its 81 provinces span an extraordinary range of climatic conditions — from the arid continental plateau of Central Anatolia to the humid subtropical Black Sea coast, and from the hot Mediterranean lowlands to the freezing highlands of Eastern Anatolia.

This climatic diversity means that crop productivity varies dramatically across provinces, and understanding **which weather and geographic factors drive these differences** is critical for national food security planning and agricultural policy.

This project was motivated by a simple but important question: **Can we predict whether a province's crop yield will be above or below average using weather and environmental data?** If so, which factors matter most — temperature, rainfall, drought conditions, or geographic location?

By answering this question with data, we can help farmers, policymakers, and agricultural planners make better-informed decisions about crop selection, irrigation investment, and climate adaptation strategies.

## 2. Data Source

### Primary Sources

The dataset was constructed by merging two primary Turkish government data sources:

**TÜİK (Turkish Statistical Institute) — Bitkisel Üretim İstatistikleri**  
Province-level annual crop production data including:
- Planted area (ekilen alan, dekar)
- Harvested area (hasat edilen alan, dekar)
- Total production (tonnes)
- Yield (kg/decar)

Crops covered: Wheat (Buğday), Barley (Arpa), Maize (Mısır), Sunflower (Ayçiçeği)

**MGM (Turkish State Meteorological Service) — İl İklim İstatistikleri**  
Province-level annual meteorological data including:
- Average temperature (°C)
- Total rainfall (mm/year)
- Number of rainy days
- Relative humidity (%)
- Climate normals (1991–2020 baseline)

### Data Collection Process

Data from both sources was compiled at the **province–year level** for all 81 Turkish provinces across the period **2012–2024** (13 years). The two datasets were merged using province name and year as join keys, with careful handling of naming inconsistencies between sources (e.g., "Afyonkarahisar" vs. "Afyon").

### Feature Engineering (Enrichment)

To go beyond raw data, I derived several additional features:

| Derived Feature | Description | Formula |
|----------------|-------------|---------|
| Drought Stress Index | Ratio of rainfall to temperature; lower values = higher drought risk | rainfall / (temperature + 1) |
| Rainfall Deviation (z-score) | How far a province's rainfall deviates from its own long-term average | (rainfall − mean) / std |
| Seasonal Temperature Amplitude | Temperature range between summer max and winter min, by region | Region-based lookup |
| Year-over-Year Yield Change | Percentage change in yield from the previous year | (yield_t − yield_{t-1}) / yield_{t-1} × 100 |
| Harvest Ratio | Proportion of planted area successfully harvested | harvested_area / planted_area |

### Target Variable

The target variable is **binary**: for each crop type, yields above the median are labeled as **High (1)**, and yields below the median are labeled as **Low (0)**. This creates a balanced classification problem (approximately 50/50 split).

### Final Dataset Summary

| Property | Value |
|----------|-------|
| Total observations | 4,212 |
| Provinces | 81 |
| Crops | 4 |
| Years | 13 (2012–2024) |
| Total features | 20 (9 numerical + 3 categorical + derived) |
| Class balance | 50.0% High / 50.0% Low |

## 3. Data Analysis

The analysis followed four stages: data cleaning, exploratory data analysis (EDA), hypothesis testing, and machine learning classification.

### 3.1 Data Cleaning

The data cleaning process (`dataCleaning.ipynb`) involved:
- Checking for missing values (none found)
- Validating data types and ranges (no negative yields, no impossible temperatures)
- Verifying that harvested area never exceeds planted area
- Creating derived features and the binary target variable
- Saving the final merged dataset as a single CSV file

### 3.2 Exploratory Data Analysis (EDA)

The EDA phase (`EDA/eda.ipynb`) produced 8 visualizations to understand the data:

**Target Variable Distribution** — The classes are well-balanced (~50/50), meaning no resampling techniques like SMOTE are needed.

![Target Distribution](EDA/01_target_distribution.png)

**Yield Distribution by Crop** — Each crop has a distinct yield range. Maize yields are highest (500–1100 kg/decar), while wheat, barley, and sunflower cluster in lower ranges. All distributions are approximately normal.

![Yield by Crop](EDA/02_yield_distribution_by_crop.png)

**Regional Yield Comparison** — Significant differences exist across Turkey's 7 geographical regions. The Akdeniz (Mediterranean) and Marmara regions consistently produce higher wheat yields, while Doğu Anadolu (Eastern Anatolia) shows the lowest yields.

![Yield by Region](EDA/03_yield_by_region_boxplot.png)

**Weather Variable Distributions** — Temperature ranges from ~3°C (Eastern Anatolia) to ~20°C (Mediterranean). Rainfall varies from ~200 mm to over 2,500 mm. This extreme diversity makes Turkey an excellent case study for weather–yield relationships.

![Weather Distributions](EDA/04_weather_distributions.png)

**Correlation Matrix** — Individual correlations between weather variables and yield are moderate (drought index: −0.16, temperature: +0.13). This indicates **non-linear, interaction-based** relationships — a key motivation for using tree-based models.

![Correlation Heatmap](EDA/05_correlation_heatmap.png)

**Weather vs. Yield Scatter Plots** — The relationship between rainfall and wheat yield is not linear: both very low and very high rainfall hurt yields, suggesting an optimal range. This confirms the non-linear hypothesis.

![Weather vs Yield](EDA/06_weather_vs_yield_scatter.png)

**Yearly Trends** — Average yields remain relatively stable across years, with minor weather-driven fluctuations. No strong long-term trend is observed.

![Yearly Trends](EDA/07_yearly_yield_trends.png)

**Region × Year Heatmap** — Persistent regional differences are visible year after year. Akdeniz consistently outperforms; Doğu Anadolu and Karadeniz consistently underperform.

![Region Year Heatmap](EDA/08_region_year_heatmap.png)

### 3.3 Hypothesis Testing

Four statistical tests were conducted (`HypothesisTesting/hypothesis_tests.ipynb`) to evaluate the significance of observed patterns:

**Test 1: Independent Samples t-Test**

*Question:* Do provinces with above-median rainfall produce significantly higher wheat yields?

| Group | n | Mean Yield | Std |
|-------|---|------------|-----|
| High Rainfall | 527 | ~245 | — |
| Low Rainfall | 526 | ~257 | — |

- Welch t-test: **p < 0.001 → H₀ REJECTED**
- Surprising finding: high-rainfall provinces showed *lower* yields, suggesting excessive rainfall is detrimental (waterlogging, disease)

![t-test](HypothesisTesting/09_ttest_rainfall_yield.png)

**Test 2: One-Way ANOVA**

*Question:* Is there a significant difference in wheat yield across 7 geographical regions?

- F = 230.77, **p ≈ 0 → H₀ REJECTED**
- Very strong regional differences confirmed

![ANOVA](HypothesisTesting/10_anova_regions.png)

**Test 3: Chi-Square Test of Independence**

*Question:* Is there a significant association between region and yield class?

- χ² significant, **p ≈ 0 → H₀ REJECTED**
- Cramér's V = 0.76 (very strong effect size)

![Chi-Square](HypothesisTesting/11_chi_square_region_yield.png)

**Test 4: Pearson Correlation**

*Question:* Is there a significant linear correlation between temperature and wheat yield?

- r = 0.32, **p < 0.001 → H₀ REJECTED**
- Moderate positive correlation

### 3.4 Machine Learning Classification

Three classification models were trained and compared (`MachineLearning/ml_classification.ipynb`):

**Preprocessing Pipeline:**
- Numerical features → `StandardScaler` (z-score normalization)
- Categorical features (region, crop, elevation) → `OneHotEncoder`
- Combined using `ColumnTransformer` inside a `Pipeline` to prevent data leakage

**Train/Test Split:** 80% training (3,369 samples), 20% test (843 samples), stratified

**Cross-Validation:** 5-fold stratified CV for all evaluations

**Hyperparameter Tuning:** `GridSearchCV` optimizing F1-score

| Model | Tuned Parameters |
|-------|-----------------|
| Logistic Regression | C=1, L1 penalty, liblinear solver |
| Random Forest | 200 trees, max_depth=20, min_samples_split=5 |
| XGBoost | 100 trees, max_depth=5, learning_rate=0.1 |

**Test Set Results:**

| Model | Accuracy | F1-Score | ROC-AUC |
|-------|----------|----------|---------|
| Logistic Regression | 0.8814 | 0.8845 | 0.9610 |
| **Random Forest** | **0.9146** | **0.9161** | 0.9705 |
| **XGBoost** | **0.9146** | 0.9155 | **0.9750** |

![Confusion Matrices](MachineLearning/12_confusion_matrices.png)

**ROC Curve Comparison:** All models achieve excellent AUC (>0.96). XGBoost leads with 0.975.

![ROC Curves](MachineLearning/13_roc_curves.png)

**Feature Importance:** The most important predictors are:
1. Seasonal Temperature Amplitude — strongest by far
2. Drought Stress Index
3. Regional indicators (Karadeniz, Doğu Anadolu, Marmara)

![Feature Importance](MachineLearning/14_feature_importance.png)

![Model Comparison](MachineLearning/15_model_comparison.png)

## 4. Findings

### Key Results

**1. Tree-based models significantly outperform linear models.**  
Random Forest and XGBoost both achieve ~91.5% accuracy and >0.97 AUC, compared to Logistic Regression's 88.1% accuracy and 0.96 AUC. This confirms that the relationship between weather conditions and crop yield is fundamentally **non-linear** — there are interaction effects and threshold behaviors that linear models cannot capture.

**2. Geographic region is the most powerful structural predictor.**  
The ANOVA test showed highly significant differences across regions (F=230.77, p≈0), and the Chi-square test revealed a very strong association between region and yield class (Cramér's V=0.76). In the ML models, regional indicator variables ranked among the top features. This makes sense: region captures not just climate but also soil quality, agricultural infrastructure, irrigation access, and farming practices.

**3. Seasonal temperature amplitude is the single strongest feature.**  
Both Random Forest and XGBoost ranked this feature first. Provinces with extreme temperature swings between summer and winter (like Eastern Anatolia with ~32°C range) consistently produce lower yields than provinces with moderate temperature ranges (like the Black Sea coast with ~18°C range). This suggests that **temperature stability matters more than average temperature** for crop productivity.

**4. Drought stress index is the most important weather-derived feature.**  
The ratio of rainfall to temperature emerged as more predictive than either variable alone, validating our feature engineering strategy. This composite feature captures the interaction between heat and moisture that drives plant water stress.

**5. Excessive rainfall hurts yields.**  
Contrary to the naive assumption that "more rain = better crops," the t-test revealed that provinces with above-average rainfall actually produced *lower* wheat yields. This is consistent with agricultural science: excessive rainfall causes waterlogging, nutrient leaching, and promotes fungal diseases. The scatter plots confirmed a non-linear relationship with an optimal rainfall range.

**6. The models generalize well.**  
Cross-validation showed small train-test performance gaps, indicating that the models capture genuine patterns rather than overfitting to noise. The 91.5% test accuracy is achieved consistently across folds.

## 5. Limitations and Future Work

### Limitations

**Spatial resolution.** Province-level annual averages are a coarse granularity. A province like Antalya has both coastal lowlands and mountainous highlands with very different microclimates. District-level (ilçe) or even farm-level data would capture this within-province variation and likely improve predictions.

**Temporal resolution.** Annual average temperature and total rainfall miss critical seasonal patterns. Crop yields are most sensitive to weather during specific growth stages (e.g., flowering, grain filling). Monthly or weekly weather data aligned with crop phenology would be more informative.

**Missing agricultural variables.** The dataset does not include irrigation coverage, fertilizer and pesticide application rates, soil type and quality, or farming technology adoption — all of which significantly affect yields. TÜİK's Agricultural Holdings Survey and the Ministry of Agriculture's records could provide some of these.

**Static regional features.** The seasonal temperature amplitude and elevation category are static by region, which inflates their importance in the models. Time-varying versions of these features would provide a fairer comparison with weather variables.

### Future Work

**1. Incorporate satellite data.** Remote sensing indices like NDVI (Normalized Difference Vegetation Index) from NASA/ESA satellites could provide real-time crop health indicators at much finer spatial resolution than province-level statistics.

**2. Add soil and irrigation data.** Integrating soil maps from Turkey's General Directorate of Rural Services and irrigation data from DSİ (State Hydraulic Works) would capture critical agricultural infrastructure effects.

**3. Multi-class or regression formulation.** Instead of binary classification (high/low), predicting actual yield values (regression) or multiple yield categories (low/medium/high) could provide more actionable insights for farmers.

**4. Temporal modeling.** Time-series approaches (LSTM, temporal convolutional networks) could capture year-to-year patterns and lagged effects (e.g., how last year's drought affects this year's soil moisture).

**5. Climate change scenarios.** Using climate projection data from IPCC models, the trained ML models could estimate how Turkish agricultural productivity might shift under different warming scenarios — providing valuable input for national adaptation planning.

## References

- TÜİK (Turkish Statistical Institute). Bitkisel Üretim İstatistikleri. https://data.tuik.gov.tr/Kategori/GetKategori?p=Tarim-111
- MGM (Turkish State Meteorological Service). İllerimize Ait Genel İstatistik Verileri. https://www.mgm.gov.tr/veridegerlendirme/il-ve-ilceler-istatistik.aspx
- FAO (Food and Agriculture Organization). FAOSTAT Crop Production Data. https://www.fao.org/faostat/en/#data/QCL
- Scikit-learn Documentation. https://scikit-learn.org/stable/
- XGBoost Documentation. https://xgboost.readthedocs.io/

---

### AI Tool Disclosure
This project used Claude (Anthropic) as an AI assistant for code generation and analysis support, as permitted by the course academic integrity policy. Full details including specific prompts and outputs are documented in `AI_USAGE.md`.